## fbfft Implementation

- `fbfft` is a CUDA-optimized FFT implementation designed for batched small 2-D FFTs (like CNN feature maps, or your CIFAR-10 images).

- It was integrated into Torch (Lua) as part of fbcuda/fbcunn.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
transform=transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
train_set=torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

In [ ]:
train_loader=torch.utils.data.DataLoader(
    train_set,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [ ]:
images, labels=next(iter(train_loader)) # Shape: (64, 3, 32, 32)

In [ ]:
device=torch.device('mps' if torch.backends.mps.is_built()
                    else 'cuda' if torch.cuda.is_available()
                    else 'cpu')

images=images.to(device)

In [ ]:
freq_images=torch.fft.fft2(images)

In [ ]:
print(freq_images.shape)

In [ ]:
# Shift zero frequency component to center of visualization
freq_shifted_images=torch.fft.fftshift(freq_images, dim=(-2, -1))

#### 🔹 Role of `dim=(-2, -1)`

* `dim` tells **which axes to shift**.
* `(-2, -1)` means:

  * `-1` → last dimension (width, W)

  * `-2` → second-to-last dimension (height, H)
* These are the **spatial dimensions** of your image (the frequency layout).

So essentially:

* `fftshift(..., dim=(-2, -1))` **shifts along the 2D spatial plane** (height & width).
* This leaves the **batch dimension** (0) and **channel dimension** (1) unchanged.

---

#### 🔹 Example

Suppose you have an FFT of one RGB CIFAR-10 image:

* Shape: `(3, 32, 32)` → (channels, height, width).
* After `fft2`, low-frequency info is at the top-left of each `(32×32)` plane.
* Applying:

  ```python
  torch.fft.fftshift(freq, dim=(-2, -1))
  ```

  moves the low frequencies to the **center of the 32×32 grid**, making it look symmetric for visualization.

---

👉 In short:
`dim=(-2, -1)` tells `fftshift` to only rearrange the **2D frequency axes (height & width)**, not the batch or channel dimensions.

- `torch`: (c, h, w)

- `matplotlib`: (h, w, c)

In [ ]:
magnitude_spectrum=torch.log(torch.abs(freq_shifted_images)+1e-8)

In [ ]:
def show_image_and_spectrum(img_tensor, spectrum_tensor):
    img=img_tensor.permute(1, 2, 0).cpu().numpy()
    spectrum=spectrum_tensor.mean(0).cpu().numpy()

    plt.figure(figsize=(6, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(spectrum, cmap='gray')
    plt.title('FFT Magnitude Spectrum')
    plt.axis('off')

    plt.show()

In [ ]:
show_image_and_spectrum(images[0], magnitude_spectrum[0])

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

transform=transforms.Compose([
    transforms.ToTensor()
])

train_set=torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader=torch.utils.data.DataLoader(
    train_set,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

images, labels=next(iter(train_loader))

device=torch.device('mps' if torch.backends.mps.is_built()
                    else 'cuda' if torch.cuda.is_available()
                    else 'cpu')

images=images.to(device)

freq_images=torch.fft.fft2(images)

# Shift zero frequency component to center of visualization
freq_shifted_images=torch.fft.fftshift(freq_images, dim=(-2, -1))

magnitude_spectrum=torch.log(torch.abs(freq_shifted_images)+1e-8)

def show_image_and_spectrum(img_tensor, spectrum_tensor):
    img=img_tensor.permute(1, 2, 0).cpu().numpy()
    spectrum=spectrum_tensor.mean(0).cpu().numpy()

    plt.figure(figsize=(6, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(spectrum, cmap='gray')
    plt.title('FFT Magnitude Spectrum')
    plt.axis('off')

    plt.show()

show_image_and_spectrum(images[0], magnitude_spectrum[0])